In [1]:

# ANSCHLUSS: KATEGORISIERTE AUSWERTUNG OFFENER FRAGEN

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats
from scipy.stats import chi2_contingency



# 0) DATEIEN LADEN

data_dir = Path("") #Pfad zum richtigen Ordner eingeben
# lokal alternativ:
# data_dir = Path("data/processed")

open_item9_long = pd.read_excel(data_dir / "open_item9_long.xlsx")
open_compare2_long = pd.read_excel(data_dir / "open_compare2_long.xlsx")
estimation_long = pd.read_excel(data_dir / "estimation_long.xlsx")

compare_long_path = data_dir / "compare_long.xlsx"
compare_long = pd.read_excel(compare_long_path) if compare_long_path.exists() else pd.DataFrame()

print("open_item9_long:", open_item9_long.shape)
print("open_compare2_long:", open_compare2_long.shape)
print("estimation_long:", estimation_long.shape)
if not compare_long.empty:
    print("compare_long:", compare_long.shape)



# 1) HILFSFUNKTIONEN

def norm_text(x):
    s = str(x).strip().lower()
    s = (
        s.replace("ä", "ae")
         .replace("ö", "oe")
         .replace("ü", "ue")
         .replace("ß", "ss")
    )
    return s

def add_keyword_frames(frame_df, text_col, frames_dict, prefix):
    out = frame_df.copy()
    out["text_norm"] = out[text_col].apply(norm_text)
    out["n_tokens"] = out["text_norm"].str.findall(r"\w+").apply(len)
    for name, kws in frames_dict.items():
        pat = "|".join(map(re.escape, kws))
        out[f"{prefix}{name}"] = out["text_norm"].str.contains(pat, na=False).astype(int)
    return out

def chi2_with_cramers_v(tab):
    chi2, p, _, _ = chi2_contingency(tab)
    n = tab.to_numpy().sum()
    r, k = tab.shape
    v = np.sqrt(chi2 / (n * (min(r, k) - 1))) if n > 0 and min(r, k) > 1 else np.nan
    return chi2, p, v



# 2) ITEM 9 kategorisieren

frames_item9 = {
    "sprache_stil": [
        "stil", "sprache", "wortwahl", "ausdruck", "formulierung",
        "sprachlich", "satzbau", "schreibstil", "ton"
    ],
    "figuren_identifikation": [
        "figur", "charakter", "protagon", "person", "held",
        "identifikation", "identifizieren", "sympath", "bezug", "naehe"
    ],
    "inhalt_handlung": [
        "inhalt", "handlung", "plot", "geschichte", "ereignis",
        "verlauf", "szene", "aufbau", "spann"
    ],
    "atmosphaere_emotion": [
        "atmosphaere", "stimmung", "gefuehl", "emotional",
        "beruehr", "ruehr", "spannend", "traurig", "unheimlich"
    ],
    "verstaendlichkeit": [
        "verstaendlich", "klar", "unverstaendlich", "kompliziert", "einfach", "lesbar"
    ],
    "maerchenhaftigkeit": [
        "maerchen", "maerchenhaft", "fantastisch", "tradition", "typisch"
    ]
}

open9_cat = open_item9_long.copy()
open9_cat["Antwort_text"] = open9_cat["response_text"].astype(str).str.strip()
open9_cat = open9_cat[
    open9_cat["Antwort_text"].notna() &
    (open9_cat["Antwort_text"] != "") &
    (open9_cat["Antwort_text"].str.lower() != "nan")
].copy()

open9_cat["Gruppe"] = open9_cat["group"]
open9_cat["id"] = open9_cat["participant_id"]

open9_cat = add_keyword_frames(open9_cat, "Antwort_text", frames_item9, prefix="I9_")
item9_frame_cols = [c for c in open9_cat.columns if c.startswith("I9_")]


print("ITEM 9 – KATEGORISIERTE AUSWERTUNG")

print("Anzahl Item-9-Antworten gesamt:", len(open9_cat))
print(open9_cat.groupby("Gruppe")["Antwort_text"].count().reset_index(name="n"))

print("\nTokenlaenge Item 9 nach Gruppe:")
print(open9_cat.groupby("Gruppe")["n_tokens"].agg(["mean", "std", "median", "count"]).reset_index())

# Teilnahme auf Personenebene
open9_participation = open9_cat.groupby(["id", "Gruppe"]).size().reset_index(name="n_item9")
print("\nPersonen mit mind. einer Item-9-Antwort:")
print(open9_participation.groupby("Gruppe")["id"].count().reset_index(name="n_persons"))

# häufigste Kategorien
item9_freq = (
    open9_cat[item9_frame_cols]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
item9_freq.columns = ["Kategorie", "Anteil"]
item9_freq["Prozent"] = item9_freq["Anteil"] * 100
print("\nHaeufigste Item-9-Kategorien insgesamt:")
print(item9_freq)

# Gruppenunterschiede in Antwortlänge
A_len = open9_cat.loc[open9_cat["Gruppe"] == "A", "n_tokens"].dropna()
B_len = open9_cat.loc[open9_cat["Gruppe"] == "B", "n_tokens"].dropna()
if len(A_len) > 1 and len(B_len) > 1:
    print("\nTokenlaenge Item 9:")
    print("Welch-t p =", stats.ttest_ind(A_len, B_len, equal_var=False).pvalue)
    print("Mann-Whitney p =", stats.mannwhitneyu(A_len, B_len, alternative="two-sided").pvalue)

# Gruppenunterschiede in Kategorien
item9_group_tests = []
for col in item9_frame_cols:
    tab = pd.crosstab(open9_cat["Gruppe"], open9_cat[col])
    if tab.shape == (2, 2):
        chi2, p, v = chi2_with_cramers_v(tab)
        item9_group_tests.append({
            "Kategorie": col.replace("I9_", ""),
            "p": p,
            "Cramers_V": v,
            "A_Anteil": open9_cat.loc[open9_cat["Gruppe"] == "A", col].mean(),
            "B_Anteil": open9_cat.loc[open9_cat["Gruppe"] == "B", col].mean()
        })

item9_group_tests = pd.DataFrame(item9_group_tests).sort_values("p")
print("\nItem 9 – Gruppenunterschiede in Kategorien:")
print(item9_group_tests)



# 3) V2 kategorisieren

frames_v2 = {
    "sprache_stil": [
        "stil", "sprache", "wortwahl", "ausdruck", "formulierung",
        "sprachlich", "satzbau", "klang", "ton"
    ],
    "verstaendlichkeit": [
        "verstaendlich", "klar", "unverstaendlich", "kompliziert", "einfach", "lesbar"
    ],
    "atmosphaere_emotion": [
        "atmosphaere", "stimmung", "gefuehl", "emotional", "beruehr",
        "ruehr", "spannend", "lebendig", "mitreiss"
    ],
    "logik_plausibilitaet": [
        "logisch", "plausibel", "unlogisch", "widerspruch", "nachvollziehbar"
    ],
    "maerchenhaftigkeit": [
        "maerchen", "maerchenhaft", "tradition", "typisch", "fantastisch"
    ],
    "kuenstlichkeit_ki": [
        "ki", "kuenstlich", "chatgpt", "generiert", "typisch ki", "maschinell"
    ]
}

vf2_cat = open_compare2_long.copy()
vf2_cat["vf2_text"] = vf2_cat["response_text"].astype(str).str.strip()
vf2_cat = vf2_cat[
    vf2_cat["vf2_text"].notna() &
    (vf2_cat["vf2_text"] != "") &
    (vf2_cat["vf2_text"].str.lower() != "nan")
].copy()

vf2_cat["Gruppe"] = vf2_cat["group"]
vf2_cat["id"] = vf2_cat["participant_id"]
vf2_cat["Textpaar"] = vf2_cat["pair_id"]

vf2_cat = add_keyword_frames(vf2_cat, "vf2_text", frames_v2, prefix="V2_")
vf2_frame_cols = [c for c in vf2_cat.columns if c.startswith("V2_")]


print("V2 – KATEGORISIERTE AUSWERTUNG")

print("Anzahl V2-Begruendungen gesamt:", len(vf2_cat))
print(vf2_cat.groupby("Gruppe")["vf2_text"].count().reset_index(name="n"))

print("\nTokenlaenge V2 nach Gruppe:")
print(vf2_cat.groupby("Gruppe")["n_tokens"].agg(["mean", "std", "median", "count"]).reset_index())

vf2_freq = (
    vf2_cat[vf2_frame_cols]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
vf2_freq.columns = ["Kategorie", "Anteil"]
vf2_freq["Prozent"] = vf2_freq["Anteil"] * 100
print("\nHaeufigste V2-Kategorien insgesamt:")
print(vf2_freq)

# Gruppenunterschiede in Kategorien
vf2_group_tests = []
for col in vf2_frame_cols:
    tab = pd.crosstab(vf2_cat["Gruppe"], vf2_cat[col])
    if tab.shape == (2, 2):
        chi2, p, v = chi2_with_cramers_v(tab)
        vf2_group_tests.append({
            "Kategorie": col.replace("V2_", ""),
            "p": p,
            "Cramers_V": v,
            "A_Anteil": vf2_cat.loc[vf2_cat["Gruppe"] == "A", col].mean(),
            "B_Anteil": vf2_cat.loc[vf2_cat["Gruppe"] == "B", col].mean()
        })

vf2_group_tests = pd.DataFrame(vf2_group_tests).sort_values("p")
print("\nV2 – Gruppenunterschiede in Kategorien:")
print(vf2_group_tests)

# 4) Präferenzlabel aus compare_long ableiten

if not compare_long.empty:
    # Frage 1 der Vergleichsfragen nutzen
    pref_v1 = compare_long[compare_long["question_no"] == 1].copy()
    pref_v1 = pref_v1[["participant_id", "group", "pair_id", "response_num"]].rename(
        columns={"response_num": "V1_rating"}
    )

    def human_pos_for_pair(pair_id):
        # Paar 1 und 4: Text 1 = Mensch
        # Paar 2 und 3: Text 2 = Mensch
        return 1 if pair_id in [1, 4] else 2

    pref_v1["Human_pos"] = pref_v1["pair_id"].apply(human_pos_for_pair)

    def pref_label_from_v1(v1, human_pos):
        if pd.isna(v1) or pd.isna(human_pos) or v1 == 4:
            return np.nan
        chosen = 1 if v1 < 4 else 2
        return "Mensch" if chosen == human_pos else "KI"

    pref_v1["Pref_label"] = [
        pref_label_from_v1(v1, hp)
        for v1, hp in zip(pref_v1["V1_rating"], pref_v1["Human_pos"])
    ]

    vf2_pref = vf2_cat.merge(
        pref_v1[["participant_id", "group", "pair_id", "Pref_label"]],
        on=["participant_id", "group", "pair_id"],
        how="left"
    )

    vf2_pref_tests = []
    sub_pref = vf2_pref.dropna(subset=["Pref_label"]).copy()

    for col in vf2_frame_cols:
        tab = pd.crosstab(sub_pref["Pref_label"], sub_pref[col])
        if tab.shape == (2, 2):
            chi2, p, v = chi2_with_cramers_v(tab)
            vf2_pref_tests.append({
                "Kategorie": col.replace("V2_", ""),
                "p": p,
                "Cramers_V": v,
                "Mensch_Anteil": sub_pref.loc[sub_pref["Pref_label"] == "Mensch", col].mean(),
                "KI_Anteil": sub_pref.loc[sub_pref["Pref_label"] == "KI", col].mean()
            })

    vf2_pref_tests = pd.DataFrame(vf2_pref_tests).sort_values("p")
    print("\nV2 – Zusammenhang Kategorien x Praeferenz:")
    print(vf2_pref_tests)

# 5) Kommentare zu Einschätzungsfragen kategorisieren

frames_comments = {
    "sprache_stil": [
        "stil", "sprache", "wortwahl", "formulierung", "ausdruck", "sprachlich", "satzbau"
    ],
    "verstaendlichkeit": [
        "verstaendlich", "klar", "unverstaendlich", "kompliziert", "einfach", "lesbar"
    ],
    "atmosphaere_emotion": [
        "atmosphaere", "stimmung", "gefuehl", "emotional", "lebendig", "schoen", "beruehr", "spann"
    ],
    "logik_plausibilitaet": [
        "logisch", "plausibel", "unlogisch", "widerspruch", "nachvollziehbar"
    ],
    "maerchenhaftigkeit": [
        "maerchen", "maerchenhaft", "typisch", "fantastisch", "sage", "legende"
    ],
    "ki_attrib": [
        "ki", "kuenstlich", "generiert", "typisch ki", "maschinell", "chatgpt"
    ]
}

comments_long = estimation_long.copy()
comments_long = comments_long[comments_long["comment"].notna()].copy()
comments_long["Kommentar"] = comments_long["comment"].astype(str).str.strip()
comments_long = comments_long[
    comments_long["Kommentar"].notna() &
    (comments_long["Kommentar"] != "") &
    (comments_long["Kommentar"].str.lower() != "nan")
].copy()

comments_long["Gruppe"] = comments_long["group"]
comments_long["id"] = comments_long["participant_id"]
comments_long["Correct"] = comments_long["correct"]

decision_comments = add_keyword_frames(comments_long, "Kommentar", frames_comments, prefix="C_")
comment_frame_cols = [c for c in decision_comments.columns if c.startswith("C_")]


print("EINSCHAETZUNGSKOMMENTARE – KATEGORISIERTE AUSWERTUNG")
print("Anzahl Entscheidungskommentare gesamt:", len(decision_comments))
print("Von insgesamt moeglichen Entscheidungen:", len(estimation_long))
print("Anteil:", len(decision_comments) / len(estimation_long))

print("\nNach Gruppe:")
print(decision_comments.groupby("Gruppe")["Kommentar"].count().reset_index(name="n"))

print("\nTokenlaenge gesamt:")
print(decision_comments["n_tokens"].describe())

print("\nTokenlaenge nach richtig/falsch:")
print(decision_comments.groupby("Correct")["n_tokens"].agg(["mean", "std", "median", "count"]).reset_index())

comment_freq = (
    decision_comments[comment_frame_cols]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
comment_freq.columns = ["Kategorie", "Anteil"]
comment_freq["Prozent"] = comment_freq["Anteil"] * 100
print("\nHaeufigste Kategorien in Entscheidungskommentaren:")
print(comment_freq)

comment_correct_summary = (
    decision_comments.groupby("Correct")[comment_frame_cols]
    .mean()
    .T
    .reset_index()
)
comment_correct_summary.columns = ["Kategorie", "falsch_0", "richtig_1"]
print("\nKategorien richtig vs falsch:")
print(comment_correct_summary.sort_values("richtig_1", ascending=False))



open_item9_long: (373, 10)
open_compare2_long: (384, 9)
estimation_long: (552, 9)
compare_long: (2592, 9)
ITEM 9 – KATEGORISIERTE AUSWERTUNG
Anzahl Item-9-Antworten gesamt: 373
  Gruppe    n
0      A  185
1      B  188

Tokenlaenge Item 9 nach Gruppe:
  Gruppe       mean       std  median  count
0      A  13.751351  12.42074    10.0    185
1      B  12.276596  11.57390     9.0    188

Personen mit mind. einer Item-9-Antwort:
  Gruppe  n_persons
0      A         43
1      B         42

Haeufigste Item-9-Kategorien insgesamt:
                   Kategorie    Anteil    Prozent
0            I9_sprache_stil  0.190349  19.034853
1         I9_inhalt_handlung  0.123324  12.332440
2       I9_verstaendlichkeit  0.104558  10.455764
3     I9_atmosphaere_emotion  0.072386   7.238606
4      I9_maerchenhaftigkeit  0.058981   5.898123
5  I9_figuren_identifikation  0.056300   5.630027

Tokenlaenge Item 9:
Welch-t p = 0.2364213617383622
Mann-Whitney p = 0.2898219780725124

Item 9 – Gruppenunterschiede in